# Welcome to Bridging Research-Grade Models and Consumer Wearables: A Personalized Federated Learning Approach to Stress Prediction 

## Getting Started:

First, activate your virtual environment, if necessary, and download all necessary packages: 

In [ ]:
pip install -r requirements.txt

## Verify Installation:
Check that all core libraries are properly installed:

In [ ]:
# Test core imports
import sys

import numpy as np
import pandas as pd
import torch

print("All packages imported successfully!")
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.version}")
print(f"NumPy: {np.version}")
print(f"Pandas: {pd.__version__}")
print(f"\nDevice: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Installation
### Option A: Automatic Download (Linux/MacOS/Bash present) <br>
If you're on Linux or Mac or Git Bash is available, run the cell below to automatically download and extract the WESAD dataset:

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

data_dir = Path("data/raw/wesad")

# If already present, skip
if data_dir.exists() and any(data_dir.iterdir()):
    print(f"✅ Dataset already present at: {data_dir.resolve()}")
else:
    # try bash if it exists (Linux/macOS/Git Bash/WSL)
    bash = shutil.which("bash")
    sh_script = Path("scripts/download_dataset.sh")

    if bash and sh_script.exists():
        subprocess.run([bash, str(sh_script)], check=True)
        print("Dataset downloaded via bash script!")
    else:
        print(" 'bash' not found (likely native Windows).")
        print("Proceeding with manual instructions.")

### Option B: Manual Download (All Platforms)

If automatic download doesn't work:

- Download: [WESAD Dataset](https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download)
- Extract the downloaded zip file
- Place the WESAD folder in: data/raw/wesad/

- Expected structure:

```
data/
└── raw/
    └── wesad/
        └── WESAD/
            ├── S2/
            ├── S3/
            └── ...

```

## Verify Dataset Location

Check if WESAD dataset is present: 

In [ ]:
from pathlib import Path

wesad_path = Path("data/raw/wesad/WESAD")

if wesad_path.exists():
    subjects = sorted(
        [d.name for d in wesad_path.iterdir() if d.is_dir() and d.name.startswith("S")]
    )
    print("WESAD dataset found!")
    print(f"Location: {wesad_path.absolute()}")
    print(f"Subjects ({len(subjects)}): {', '.join(subjects)}")
else:
    print("WESAD dataset NOT found!")
    print(f"Expected at: {wesad_path.absolute()}")
    print("\nPlease download and place the dataset in the correct location.")

## Configuration Setup
Set up your environment configuration:

In [ ]:
# Create .env file if it doesn't exist

from pathlib import Path

from dotenv import load_dotenv

env_file = Path(".env")

if not env_file.exists():
    env_example = Path(".env.example")

    if env_example.exists():
        with open(env_example) as f:
            content = f.read()

        with open(env_file, "w") as f:
            f.write(content)

        print("Created .env from .env.example")

    else:
        print("ℹ️ No .env.example found, using defaults")

else:
    print("ℹ️ .env already exists")

# Load environment variables
load_dotenv()
print("Configuration loaded")

You can now proceed with:

Data Preprocessing - Process raw WESAD signals
Feature Extraction - Extract time/frequency domain features
Model Training - Train baseline and federated models
Evaluation - Analyze results

Continue to the next section below to start preprocessing the data.

## Data Preprocessing

### Load and Merge WESAD Dataset

During this step, the biosensor data from the WESAD directory will be automatically extracted into several merged files in `.pkl` format.<br>

This may take a few minutes depending on the machine.

In [ ]:
from src.data import WESADResampleMerge

print("Initializing WESAD preprocessor...")
preprocessor = WESADResampleMerge(
    dataset_path="data/raw/wesad/WESAD/",
    IDs=[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17],
)

# Loads metadata, merges wrist data (resample to BVP rate),chest data and filters labels and invalid temperatures
print("\nStarting preprocessing pipeline...")
preprocessor.resample_merge()

# Load the merged data
print("\nLoading merged datasets...")
chest_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_chest.pkl")
wrist_df = pd.read_pickle("data/raw/wesad/WESAD/merged/merged_wrist.pkl")

# Display info
print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print("\nChest Data:")
print(f"  Shape: {chest_df.shape}")
print(f"  Columns: {list(chest_df.columns)}")
print(f"  Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nWrist Data:")
print(f"  Shape: {wrist_df.shape}")
print(f"  Columns: {list(wrist_df.columns)}")
print(f"  Memory: {wrist_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\nLabel Distribution (Chest):")
print(chest_df["label"].value_counts().sort_index())

### Data Validation

In [ ]:
print("QUICK DATA VALIDATION CHECKS:\n")

#  Basic Info
print("\n1. CHEST DATA")
print(f"   Shape: {chest_df.shape}")
print(f"   Memory: {chest_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(chest_df.columns)}")

print("\n2. WRIST DATA")
print(f"   Shape: {wrist_df.shape}")
print(f"   Memory: {wrist_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Columns: {list(wrist_df.columns)}")

print("\n3. LABEL DISTRIBUTION")
print("\n   Chest:")
print(chest_df["label"].value_counts().sort_index())
print("\n   Wrist:")
print(wrist_df["label"].value_counts().sort_index())

# Check for Invalid Temps (discovered in EDA notebook 2)
print("\n4. TEMPERATURE VALIDATION")
chest_temp_col = "c_temp" if "c_temp" in chest_df.columns else "temp"
wrist_temp_col = "w_temp"

chest_invalid = (chest_df[chest_temp_col] <= 0).sum()
wrist_invalid = (wrist_df[wrist_temp_col] <= 0).sum()

print(f"   Chest temp ≤0: {chest_invalid} (should be 0)")
print(f"   Wrist temp ≤0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" Temperature interpolation successful!")
else:
    print(" Warning: Still have invalid temperatures")

# Check for Invalid eda values (discovered in EDA notebook 2)
print("\n5. EDA VALIDATION")
chest_eda_col = "c_eda" if "c_eda" in chest_df.columns else "eda"
wrist_eda_col = "w_eda"

chest_invalid = (chest_df[chest_eda_col] < 0).sum()
wrist_invalid = (wrist_df[wrist_eda_col] < 0).sum()

print(f"   Chest eda <0: {chest_invalid} (should be 0)")
print(f"   Wrist eda <0: {wrist_invalid} (should be 0)")

if chest_invalid == 0 and wrist_invalid == 0:
    print(" EDA interpolation successful! No negative artifacts.")
else:
    print(" Warning: Still have invalid eda artifacts")


print("\n6. MISSING DATA")
chest_nans = chest_df.isnull().sum()
wrist_nans = wrist_df.isnull().sum()

if chest_nans.sum() > 0:
    print(f"   Chest NaNs:\n{chest_nans[chest_nans > 0]}")
else:
    print("   Chest: No missing values")

if wrist_nans.sum() > 0:
    print(f"   Wrist NaNs:\n{wrist_nans[wrist_nans > 0]}")
else:
    print("   Wrist: No missing values")

print("\nData validation complete!")